# 🌦️ January 2022 — Multi-city Weather Comparison (Radar) — AccuWeather (Historical Daily)
**Provider:** AccuWeather · **Source:** Databricks Marketplace via Delta Sharing · **Runtime:** DBR 16.4 LTS

Compares **January 2022** across U.S. cities on a **radar chart**, using five metrics aggregated over the month:
average temperature, total precipitation, average snow depth, **average wind speed**, and **average visibility**.

Only the **“superlative” cities** are kept on the radar — the **snowiest**, the **hottest**, the **windiest**,
the **brightest** (by solar irradiance) and the one with the **best visibility**.

> **Confirm against your installed share** (all in the `CONFIG` cell):
> - `TABLE` — catalog/schema/table created on *Get instant access*.
> - Column names via `df_raw.printSchema()` — especially the postal-code key and the metric columns.
> - **Units** — assumes **SI**: temperature °C, precipitation & snow in **mm**. If your table is Imperial, values are °F / inches.
> - `CITY_ZIPS` — the postal codes below are representative downtown ZIPs; replace with codes actually present in the sample (Section 2 lists them).

## 1- Setup

In [0]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

### Config — the only cell you should need to edit

In [0]:
# --- 1. Shared table (replace after installing the Marketplace share) ----------
TABLE = "accuweather_historical_weather_data_u_s_postal_codes_sample.historical.us_postal_daily_metric"

# --- 2. Column names (confirm with df_raw.printSchema()) -----------------------
POSTAL_COL = "POSTAL_CODE"                 # postal-code / ZIP key column
DATE_COL   = "DATE_CALENDAR"               # daily date column
TEMP_COL   = "TEMPERATURE_AVG"             # avg air temperature (SI: °C)
PRECIP_COL = "PRECIPITATION_LWE_TOTAL"     # total precip (SI: mm)
SNOW_COL   = "SNOW_DEPTH_AVG"              # avg snow depth (SI: mm)  [alt: SNOW_TOTAL for snowfall]
WIND_COL   = "WIND_SPEED_AVG"             # avg wind speed   (replaces the old cloud metric)
VIS_COL    = "VISIBILITY_AVG"             # avg visibility   (replaces the old ice metric)
SUN_COL    = "SOLAR_IRRADIANCE_AVG"       # avg solar irradiance -> used to pick the "brightest" city
                                          #   [alt: MINUTES_OF_SUN_TOTAL for sunshine duration]

# --- 3. Period -----------------------------------------------------------------
YEAR, MONTH = 2022, 1                       # the sample only covers January 2022

# --- 4. Cities of interest (replace ZIPs with codes present in the sample) ------
CITY_ZIPS = {
    "New York":      "10001",
    "Chicago":       "60601",
    "New Orleans":   "70112",
    "Los Angeles":   "90012",
    "San Francisco": "94103",
    "Washington":    "20001",
    "Dallas":        "75201",
    "Miami":         "33101",
    "Las Vegas":     "89101",
}
CITIES = list(CITY_ZIPS.keys())

## 2- Load from Delta Sharing

In [0]:
df_raw = spark.table(TABLE)
df_raw.printSchema()

In [0]:
# Sanity check: the sample should be January 2022, and here are the postal codes it contains
display(df_raw.select(F.min(DATE_COL).alias("min_date"), F.max(DATE_COL).alias("max_date")))
display(df_raw.select(POSTAL_COL).distinct().orderBy(POSTAL_COL))

## 3- Prepare & aggregate (January 2022, one row per city)
Keep January 2022 and our nine postal codes, attach a city label, derive the two day-level flags
(cloudy day, ice day), then aggregate each metric over the month.

In [0]:
df_month = (
    df_raw
    .withColumn("year",  F.year(DATE_COL))
    .withColumn("month", F.month(DATE_COL))
    .filter((F.col("year") == YEAR) & (F.col("month") == MONTH))
    .filter(F.col(POSTAL_COL).isin(list(CITY_ZIPS.values())))
    .withColumn(
        "city",
        F.create_map(*[item for pair in
            [(F.lit(zip_code), F.lit(city)) for city, zip_code in CITY_ZIPS.items()]
            for item in pair
        ])[F.col(POSTAL_COL)]
    )
)

agg = (
    df_month
    .groupBy("city", F.col(POSTAL_COL).alias("postal_code"))
    .agg(
        F.round(F.avg(F.col(TEMP_COL).cast("double")),   1).alias("temp_avg"),
        F.round(F.sum(F.col(PRECIP_COL).cast("double")), 1).alias("precip_total"),
        F.round(F.avg(F.col(SNOW_COL).cast("double")),   1).alias("snow_depth_avg"),
        F.round(F.avg(F.col(WIND_COL).cast("double")),   1).alias("wind_speed_avg"),
        F.round(F.avg(F.col(VIS_COL).cast("double")),    1).alias("visibility_avg"),
        F.round(F.avg(F.col(SUN_COL).cast("double")),    1).alias("sun_avg"),
    )
    .orderBy("city")
)
display(agg)   # <-- raw values (the radar below is normalised, so keep this table for magnitudes)

## 4- Radar chart
Each axis is **min–max normalised across the nine cities** (0 = lowest, 1 = highest), because the raw
metrics live on very different scales. Read the shape for relative comparison; read Section 3's table for absolute values.

In [0]:
# --- 1. Keep only the "superlative" cities -------------------------------------
# One city per criterion. Radar axes are temp / precip / snow / wind / visibility,
# while "brightest" is picked on solar irradiance (sun_avg) even though it is not an axis.
selection = {
    "snowiest":        "snow_depth_avg",
    "hottest":         "temp_avg",
    "windiest":        "wind_speed_avg",
    "brightest":       "sun_avg",
    "best visibility": "visibility_avg",
}

pdf_all = agg.toPandas().set_index("city")

chosen = {}   # city -> list of criteria it wins
for crit, col in selection.items():
    winner = pdf_all[col].astype(float).idxmax()
    chosen.setdefault(winner, []).append(crit)

print("Cities kept on the radar:")
for city, crits in chosen.items():
    print(f"  - {city}: {', '.join(crits)}")

pdf = pdf_all.loc[list(chosen.keys())]

# --- 2. Radar axes -------------------------------------------------------------
metric_cols = ["temp_avg", "precip_total", "snow_depth_avg", "wind_speed_avg", "visibility_avg"]
labels      = ["Avg temp (°C)", "Total precip (mm)", "Snow depth (mm)", "Wind speed", "Visibility"]

# min-max normalise each axis to [0, 1] across the kept cities; constant axis -> 0
norm = pdf[metric_cols].astype(float).copy()
for c in metric_cols:
    lo, hi = norm[c].min(), norm[c].max()
    norm[c] = (norm[c] - lo) / (hi - lo) if hi > lo else 0.0

N = len(metric_cols)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
colors = plt.cm.tab10(np.linspace(0, 1, len(norm)))
for (city, row), col in zip(norm.iterrows(), colors):
    vals = row[metric_cols].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=1.8, label=city, color=col)
    ax.fill(angles, vals, alpha=0.05, color=col)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels([])
ax.set_title(f"January {YEAR} — superlative cities (each axis min–max normalised)\n", size=13)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10), fontsize=9)
display(fig)
plt.close(fig)